In [12]:
import math
class Value:
    def __init__(self,data,_children=(),_op='',label=''):
        self.data = data
        self._prev = set(_children)
        self._op = _op
        self.label = label
        self.grad = 0.0
        self._backward = lambda: None

    def __repr__(self):
        return f"value(data={self.data})"

    def __mul__(self,other):
        other = other if isinstance(other,Value) else Value(other)
        out = Value(self.data * other.data,(self,other),'*')

        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = _backward

        return out

    def __log__(self):
        out = Value(math.log(self.data),(self,),'log')

        def _backward():
            self.grad += (1/self.data) * out.grad
        out._backward = _backward

        return out

    def __pow__(self,other):
        assert isinstance(other,(int,float))
        out = Value(self.data**other,(self,),f'**{other}')

        def _backward():
            self.grad += (other * (self.data ** (other - 1))) * out.grad
        out._backward = _backward

        return out

    def log(self):
      x=self.data
      out=Value(math.log(x),(self,),'log')
      def _backward():
        self.grad += (1/x) * out.grad
      out._backward=_backward
      return out

    def sin(self):
        out=Value(math.sin(self.data),(self,),f'sin({self})')
        def _backward():
            self.grad+=math.cos(self.data)*out.grad
        out._backward=_backward
        return out

    def backward(self):
         topo = []
         visited = set()
         def build_topo(v):
           if v not in visited:
              visited.add(v)
           for child in v._prev:
              build_topo(child)
           topo.append(v)
         build_topo(self) # Fixed: Call build_topo to create the topological order
         self.grad=1.0 # Fixed: Initialize grad for the root node
    # Fixed: Loop to traverse topological order for backward pass
         for node in reversed(topo):
             node._backward()
            

In [13]:
X=Value(2.0,label='x')
term1=(X**4).log()
term1.label='log(x^4)'
term2=(X**3).sin()
term2.label='sin(x^3)'
Y=term1*term2
Y.label='y'
Y.backward()



In [ ]:
xv = 2.0
analytical_grad = (4.0/xv)*math.sin(xv**3) + 3*(xv**2)*math.cos(xv**3)*math.log(xv**4)

print(f"Forward Pass f(x) at x=2.0:   {Y.data:.4f}")
print(f"Autodiff Gradient f'(x):      {X.grad:.4f}")
print(f"Analytical Gradient f'(x):    {analytical_grad:.4f}")   

Forward Pass f(x) at x=2.0:   2.7431
Autodiff Gradient f'(x):      -2.8622
Analytical Gradient f'(x):    -2.8622
